In [1]:
import pandas as pd
from PIL import Image, ImageDraw, ImageFont
import win32clipboard
import io

df = pd.DataFrame([
    {"Agent": "Madan Das",             "ID": "CNX-001", "Time": "00:00:22", "Category": "TEAM MEETING", "Contacts": 0,  "Handle%": "0%",  "AHT": "N/A",      "CSAT": "N/A", "LOB": "Chat_AC_GLB_EN",  "Location": "Kolkata",  "Status": "Active"},
    {"Agent": "Dipanwita Chakraborty", "ID": "CNX-002", "Time": "00:12:43", "Category": "PERSONAL",     "Contacts": 0,  "Handle%": "0%",  "AHT": "N/A",      "CSAT": "N/A", "LOB": "Chat_AC_GLB_EN",  "Location": "Kolkata",  "Status": "Active"},
    {"Agent": "Rohan Mehta",           "ID": "CNX-003", "Time": "02:34:11", "Category": "INBOUND",      "Contacts": 18, "Handle%": "94%", "AHT": "00:08:33", "CSAT": "4.7", "LOB": "Chat_AC_GLB_EN",  "Location": "Mumbai",   "Status": "Active"},
    {"Agent": "Priya Sharma",          "ID": "CNX-004", "Time": "01:55:40", "Category": "OUTBOUND",     "Contacts": 12, "Handle%": "88%", "AHT": "00:09:57", "CSAT": "4.2", "LOB": "Voice_AC_GLB_EN", "Location": "Delhi",    "Status": "Active"},
    {"Agent": "Arjun Nair",            "ID": "CNX-005", "Time": "00:00:00", "Category": "ABSENT",       "Contacts": 0,  "Handle%": "0%",  "AHT": "N/A",      "CSAT": "N/A", "LOB": "Chat_AC_GLB_EN",  "Location": "Chennai",  "Status": "Inactive"},
])

# ── COLORS ───────────────────────────────────────────────────────────────────
HEADER_BG = (26,  79, 138); HEADER_FG = (255, 255, 255)
ROW_BG    = {
    "TEAM MEETING": (255, 251, 240), "PERSONAL":  (255, 251, 240),
    "INBOUND":      (240, 255, 244), "OUTBOUND":  (239, 246, 255),
    "ABSENT":       (255, 245, 245),
}
CAT_FG = {
    "TEAM MEETING": (133, 100,  4), "PERSONAL": (133, 100,  4),
    "INBOUND":      ( 10,  92, 54), "OUTBOUND": ( 10,  54,120),
    "ABSENT":       (132,  32, 41),
}
ROW_ALT = (245, 247, 250)  # zebra stripe cho row bình thường

def hex2rgb(h): return tuple(int(h[i:i+2],16) for i in (0,2,4))

def build_table_image(df):
    FONT_SIZE   = 13
    HEADER_H    = 32
    ROW_H       = 26
    PAD_X       = 10
    BORDER      = (200, 200, 200)

    try:
        font      = ImageFont.truetype("segoeui.ttf",  FONT_SIZE)
        font_bold = ImageFont.truetype("segoeuib.ttf", FONT_SIZE)
    except:
        font = font_bold = ImageFont.load_default()

    # Tính column widths
    dummy = Image.new("RGB", (1,1)); d = ImageDraw.Draw(dummy)
    col_widths = []
    for col in df.columns:
        vals = [col] + [str(v) for v in df[col]]
        w = max(d.textlength(v, font=font_bold if v==col else font) for v in vals)
        col_widths.append(int(w) + PAD_X*2)

    total_w = sum(col_widths) + 1
    total_h = HEADER_H + ROW_H * len(df) + 1

    img  = Image.new("RGB", (total_w, total_h), (255,255,255))
    draw = ImageDraw.Draw(img)

    # ── Header ───────────────────────────────────────────────────────────────
    x = 0
    for ci, col in enumerate(df.columns):
        w = col_widths[ci]
        draw.rectangle([x, 0, x+w, HEADER_H], fill=HEADER_BG)
        draw.text((x+PAD_X, HEADER_H//2 - FONT_SIZE//2), col, font=font_bold, fill=HEADER_FG)
        draw.line([(x,0),(x,HEADER_H)], fill=BORDER, width=1)
        x += w
    draw.line([(0,0),(total_w,0)],       fill=BORDER, width=1)
    draw.line([(0,HEADER_H),(total_w,HEADER_H)], fill=BORDER, width=1)
    draw.line([(total_w-1,0),(total_w-1,HEADER_H)], fill=BORDER, width=1)

    # ── Data rows ─────────────────────────────────────────────────────────────
    for ri, (_, row) in enumerate(df.iterrows()):
        y   = HEADER_H + ri * ROW_H
        cat = str(row.get("Category",""))
        bg  = ROW_BG.get(cat, ROW_ALT if ri%2 else (255,255,255))

        x = 0
        for ci, col in enumerate(df.columns):
            w   = col_widths[ci]
            val = str(row.get(col,""))
            draw.rectangle([x, y, x+w, y+ROW_H], fill=bg)

            # Text color
            if col == "Category":
                fg = CAT_FG.get(cat, (0,0,0)); f = font_bold
            elif col == "Agent":
                fg = (0,0,0); f = font_bold
            elif col == "Status":
                fg = (40,167,69) if val=="Active" else (220,53,69); f = font_bold
            elif col == "CSAT":
                try:
                    v  = float(val)
                    fg = (10,92,54) if v>=4.5 else (133,100,4) if v>=4.0 else (132,32,41)
                    val = val+" ★" if v>=4.5 else val
                except: fg=(170,170,170)
                f = font_bold
            elif col == "Handle%":
                try:
                    v  = float(val.replace('%',''))
                    fg = (10,92,54) if v>=90 else (133,100,4) if v>=80 else (170,170,170) if v==0 else (132,32,41)
                except: fg=(0,0,0)
                f = font
            elif val == "N/A":
                fg=(170,170,170); f=font
            else:
                fg=(0,0,0); f=font

            draw.text((x+PAD_X, y+ROW_H//2-FONT_SIZE//2), val, font=f, fill=fg)
            draw.line([(x,y),(x,y+ROW_H)],           fill=BORDER, width=1)
            draw.line([(x,y+ROW_H),(x+w,y+ROW_H)],   fill=BORDER, width=1)
            x += w

        draw.line([(total_w-1,y),(total_w-1,y+ROW_H)], fill=BORDER, width=1)

    return img

def img_to_clipboard(img):
    output = io.BytesIO()
    img.convert("RGB").save(output, "BMP")
    data = output.getvalue()[14:]  # strip BMP file header
    output.close()

    win32clipboard.OpenClipboard()
    win32clipboard.EmptyClipboard()
    win32clipboard.SetClipboardData(win32clipboard.CF_DIB, data)
    win32clipboard.CloseClipboard()
    print("✅ Done — Ctrl+V vào Teams!")

# pip install pillow pywin32
img = build_table_image(df)
img_to_clipboard(img)

✅ Done — Ctrl+V vào Teams!
